# Microscopy Image Analysis Basics

**A hands-on introduction using Python, NumPy, Matplotlib, SciPy, and scikit-image**

This notebook introduces the core ideas behind quantitative microscopy image analysis. The examples are deliberately small and synthetic so that you can see exactly what the computer is doing.

## Learning goals

By the end of this notebook, you should be able to:

- Explain that a digital image is an array of numerical pixel values.
- Distinguish image **data** from image **visualization**.
- Work with images that contain multiple channels, Z planes, and time points.
- Explain bit depth, dynamic range, pixel size, and spatial resolution.
- Apply basic image filters such as Gaussian blur and edge detection.
- Convert an intensity image into a binary mask using thresholding.
- Explain segmentation and simple morphological cleanup.
- Identify connected components and measure objects.
- Explain the purpose of image registration and perform a simple translation registration.
- Connect image processing operations to a reproducible quantitative workflow.

> **Teaching principle:** A microscopy image is data first and a picture second.

## 0. Setup

The notebook uses common scientific-Python packages. If they are missing on your system, install them with:

```bash
pip install numpy scipy matplotlib scikit-image pandas tifffile ipywidgets
```

`ipywidgets` is used for the interactive widgets throughout the notebook. All imports are collected in the code cell below.


In [ ]:
import io
from functools import lru_cache

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from scipy import ndimage as ndi

from skimage import color, exposure, feature, filters, measure, morphology, registration
from skimage import data
from skimage.measure import label
from skimage.restoration import rolling_ball
import tifffile

import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass

np.random.seed(7)

plt.rcParams["figure.figsize"] = (6, 5)
plt.rcParams["font.size"] = 12


## 1. A digital image is an array of numbers

A grayscale image can be represented as a 2-D array. Each array element corresponds to one pixel.

In Python, the usual indexing order is:

```text
image[y, x]
```

not `image[x, y]`.

In [ ]:
synthetic_img = np.zeros((10, 12), dtype=np.uint8)

synthetic_img[2:6, 3:7] = 80
synthetic_img[6:10, 7:12] = 180
synthetic_img[3:7, 10:12] = 255

print("Shape:", synthetic_img.shape)
print("Data type:", synthetic_img.dtype)
print("Minimum:", synthetic_img.min())
print("Maximum:", synthetic_img.max())


In [ ]:
plt.figure(figsize=(8, 5))
plt.imshow(synthetic_img, cmap="gray", interpolation="nearest")
plt.colorbar(label="Pixel intensity")
plt.xticks(range(synthetic_img.shape[1]))
plt.yticks(range(synthetic_img.shape[0]))
plt.grid(alpha=0.25)
plt.title("The image is an array of pixel values")
plt.show()


In [ ]:
synthetic_img

In [ ]:
print("Pixel at y=3, x=4:", synthetic_img[3, 4])
print("Pixel at y=8, x=10:", synthetic_img[8, 10])
print("Pixel at y=0, x=0:", synthetic_img[0, 0])


### Try it yourself

    1. Print the value of the pixel at `y=5, x=10`.
    2. Change one background pixel to 255, and display it.

In [ ]:
# 1.
# YOUR CODE HERE

# 2.
# YOUR CODE HERE


### Exercise: extract the white tower

The camera image displayed above, `skimage.data.camera()`, is a 512 × 512 grayscale
image. It contains a white tower in the background.

1. Find the white tower, then extract it as a box by supplying row and column
   coordinates.
2. Display the extracted box with `plt.imshow(cropped_image)`.

Remember: the image is indexed `image[y, x]`, so a box is `image[y_start:y_end, x_start:x_end]`.

In [ ]:
camera = data.camera()

plt.figure(figsize=(6, 6))
plt.imshow(camera, cmap="gray")
plt.xticks(np.arange(0, 512, 50))
plt.yticks(np.arange(0, 512, 50))
plt.grid(alpha=0.25)
plt.title("skimage.data.camera()")
plt.show()

In [ ]:
# Find the white tower and extract it as a box by supplying coordinates
#    (a box of rows and columns: camera[start:stop, start:stop])
# cropped_image = camera[start:stop, start:stop]

# YOUR CODE HERE

# Display the extracted box

# YOUR CODE HERE

# plt.imshow(...)

## 2. Image dimensions: X, Y, Z, channel, and time

Microscopy datasets often contain more than two dimensions. A common conceptual ordering is:

```text
T, C, Z, Y, X
```

where:

- **T** = time
- **C** = channel
- **Z** = optical section / depth
- **Y, X** = image plane

File formats and libraries may store dimensions in different orders, so always inspect metadata and array shape.

In [ ]:
# Create a tiny 5-D example: T, C, Z, Y, X
demo_5d = tifffile.imread('mitosis_TCZYX.ome.tif')

print("Shape:", demo_5d.shape)
print("T =", demo_5d.shape[0])
print("C =", demo_5d.shape[1])
print("Z =", demo_5d.shape[2])
print("Y =", demo_5d.shape[3])
print("X =", demo_5d.shape[4])


Selecting one 2-D plane means choosing one time point, one channel, and one Z plane:

In [ ]:
plane = demo_5d[0, 0, 0]

print("Selected plane shape:", plane.shape)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

img = demo_5d

# Image dimensions: (T, C, Z, Y, X)
T, C, Z, Y, X = img.shape

# Set fixed contrast limits for each channel
# (so brightness does not change artificially between timepoints)
green_min, green_max = np.percentile(img[:, 0], [1, 99.5])
red_min, red_max = np.percentile(img[:, 1], [1, 99.5])

def normalize(channel, vmin, vmax):
    """Scale intensities to [0, 1] for visualization."""
    return np.clip(
        (channel.astype(np.float32) - vmin) / max(vmax - vmin, 1e-6),
        0, 1
    )

def update(t, z):
    # Select the two channels at the chosen T and Z
    green = normalize(img[t, 0, z], green_min, green_max)
    red = normalize(img[t, 1, z], red_min, red_max)

    # Build RGB image
    rgb = np.zeros((Y, X, 3), dtype=np.float32)
    rgb[..., 0] = red
    rgb[..., 1] = green

    # Display
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(rgb)
    ax.set_title(f"Time: {t+1}/{T}    Z: {z+1}/{Z}", fontsize=15)
    ax.axis("off")
    plt.show()

# Interactive sliders
t_slider = widgets.IntSlider(
    value=0, min=0, max=T-1, step=1,
    description="Time (T):",
    continuous_update=False
)

z_slider = widgets.IntSlider(
    value=Z//2, min=0, max=Z-1, step=1,
    description="Z slice:",
    continuous_update=False
)

ui = widgets.VBox([t_slider, z_slider])
out = widgets.interactive_output(
    update, {"t": t_slider, "z": z_slider}
)

display(ui, out)

### Try it yourself

Extract the middle time point, first channel, central Z plane from `demo_5d` - and plot it!

In [ ]:
# YOUR CODE HERE
# plt.imshow(demo_5d[t_middle, c_first, z_middle])

## 3. Channels are data; colors are display choices

Fluorescence channels are independent intensity arrays. A channel is not inherently “green” or “magenta.” A lookup table, or LUT, determines how numerical values are displayed.

In [ ]:
camera = data.camera()

print("Image shape:", camera.shape)
print("Data type:", camera.dtype)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(camera, cmap="gray")
axes[0].set_title("Camera image shown in gray")

axes[1].imshow(camera, cmap="Greens")
axes[1].set_title("Same image shown in green")

axes[2].imshow(camera, cmap="magma")
axes[2].set_title("Same image shown with 'magma'")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Underlying value at [122, 427]:", camera[122, 427])

The three panels above contain the **same numerical data**. Only the mapping from intensity to screen color changed.

**Key distinction**

- Changing a LUT: visualization changes, underlying pixel values do not.
- Filtering, arithmetic, thresholding, resampling: the underlying data change.

### Try it yourself

Plot the same cameraman image with **"inferno"** and **"plasma"** colormaps. Which one reveals more details in the image?

In [ ]:
# YOUR CODE HERE
# plt.imshow(... , cmap="...")

## 4. Display range can strongly change appearance

A LUT is not the only display choice. The displayed minimum and maximum also matter.

Here, the data remain unchanged, but the contrast changes:

In [ ]:
moon = data.moon()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(moon, cmap="gray")
axes[0].set_title("Full display range")

axes[1].imshow(moon, cmap="gray", vmin=100, vmax=150)
axes[1].set_title("vmin=100, vmax=150")

axes[2].imshow(moon, cmap="gray", vmin=200, vmax=255)
axes[2].set_title("vmin=200, vmax=255")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Underlying value at [122, 427]:", moon[122, 427])

Contrast of an image can be adjusted automatically by discarding the brightest and the dimmest pixel values (which are usually not many of them). This operation changes the raw data values.

In [ ]:
# Automatic conrtrast adjustment

p_low, p_high = np.percentile(moon, [1, 99])

image_rescaled = exposure.rescale_intensity(
    moon,
    in_range=(p_low, p_high),
    out_range="dtype"
)
plt.imshow(image_rescaled, cmap="gray")

### Try it yourself

Use automatic rescaling of intensity between 2 and 98 percentile on the `camera` image

Look at the cell above for an example. Uncoment the code lines in the cell below and fill in the gaps.

In [ ]:
# YOUR CODE HERE

# p_low, p_high = ...

# image_rescaled = exposure.rescale_intensity(...)

# plt.imshow(...)

## 5. A synthetic microscopy image

From here onward, we will use a small “cell-like” synthetic fluorescence image. Because we generated it ourselves, we know what structures should exist.

In [ ]:
def add_gaussian_spot(image, x0, y0, amplitude, sigma):
    yy, xx = np.mgrid[:image.shape[0], :image.shape[1]]
    image += amplitude * np.exp(
        -((xx - x0)**2 + (yy - y0)**2) / (2 * sigma**2)
    )

microscopy = np.zeros((256, 256), dtype=np.float32)

spots = [
    (45, 55, 1.0, 8),
    (85, 80, 0.8, 11),
    (145, 65, 1.1, 9),
    (200, 90, 0.7, 14),
    (65, 155, 0.9, 12),
    (130, 165, 1.2, 10),
    (195, 180, 0.75, 13),
    (110, 215, 0.65, 8),
]

for x0, y0, amp, sigma in spots:
    add_gaussian_spot(microscopy, x0, y0, amp, sigma)

microscopy += 0.12
microscopy += np.random.normal(0, 0.055, microscopy.shape)
microscopy = np.clip(microscopy, 0, None)

plt.imshow(microscopy, cmap="gray")
plt.colorbar(label="Intensity")
plt.title("Synthetic fluorescence image")
plt.axis("off")
plt.show()


## 6. Filtering

Filters replace each pixel using information from its neighborhood.

### Gaussian blur

Gaussian filtering suppresses high-frequency noise but also smooths object boundaries.

In [ ]:
blur_1 = filters.gaussian(microscopy, sigma=1)
blur_4 = filters.gaussian(microscopy, sigma=4)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(microscopy, cmap="gray")
axes[0].set_title("Original")

axes[1].imshow(blur_1, cmap="gray")
axes[1].set_title("Gaussian sigma=1")

axes[2].imshow(blur_4, cmap="gray")
axes[2].set_title("Gaussian sigma=4")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


Larger `sigma` means stronger smoothing. Filtering can improve segmentation, but excessive smoothing can merge nearby structures or erase small ones.

### Try it yourself

Apply Gaussian filters with `sigma=0.5`, `2`, and `6`. Compare the images or their standard deviations. What happens to fine detail as sigma increases?

In [ ]:
# YOUR CODE HERE


## 7. Thresholding: from intensities to a binary mask

Thresholding asks a simple classification question for every pixel:

```text
Is this pixel above or below a chosen intensity?
```

In [ ]:
manual_threshold = 0.35
manual_mask = microscopy > manual_threshold

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(microscopy, cmap="gray")
axes[0].set_title("Intensity image")

axes[1].imshow(manual_mask, cmap="gray")
axes[1].set_title(f"Mask: intensity > {manual_threshold}")

for ax in axes:
    ax.axis("off")

plt.show()

print("Foreground pixels:", np.sum(manual_mask))
print("Foreground fraction:", np.sum(manual_mask) / (256*256))

Instead of selecting the threshold manually, algorithms such as **Otsu's method** estimate a threshold from the image histogram.

In [ ]:
otsu_threshold = filters.threshold_otsu(microscopy)
otsu_mask = microscopy > otsu_threshold

print("Otsu threshold:", otsu_threshold)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(microscopy.ravel(), bins=100)
axes[0].axvline(otsu_threshold, linestyle="--")
axes[0].set_xlabel("Intensity")
axes[0].set_ylabel("Pixel count")
axes[0].set_title("Histogram + Otsu threshold")

axes[1].imshow(otsu_mask, cmap="gray")
axes[1].set_title("Otsu mask")
axes[1].axis("off")

plt.tight_layout()
plt.show()


### Try it yourself

In the code above, replace `microscopy` image with `blur_4`. What changed? Is it easier to do the thresholding?

Try thresholds `0.25` or `0.50`. For selected threshold, calculate the number or fraction of foreground pixels. How sensitive is the result to threshold choice?

In [ ]:
# YOUR CODE HERE


## 8. Segmentation and morphological cleanup

A thresholded mask is often only the beginning of segmentation.

Common cleanup operations include:

- removing small objects,
- filling small holes,
- erosion and dilation,
- opening and closing.

In [ ]:
raw_mask = microscopy > otsu_threshold

clean_mask = morphology.remove_small_objects(raw_mask, min_size=80)
clean_mask = morphology.remove_small_holes(clean_mask, area_threshold=80)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(microscopy, cmap="gray")
axes[0].set_title("Original")

axes[1].imshow(raw_mask, cmap="gray")
axes[1].set_title("Threshold mask")

axes[2].imshow(clean_mask, cmap="gray")
axes[2].set_title("Cleaned mask")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


## 9. Connected components: turning foreground pixels into objects

A binary mask only says foreground/background. Connected-component labeling assigns a unique integer ID to each connected object.

In [ ]:
labels = measure.label(clean_mask, connectivity=2)

print("Number of connected components:", labels.max())
print("Unique labels:", np.unique(labels))


In [ ]:
plt.figure(figsize=(7, 6))
plt.imshow(labels, cmap="nipy_spectral")
plt.colorbar(label="Object label")
plt.title("Connected-component labels")
plt.axis("off")
plt.show()


## 10. Measure segmented objects

Once objects have labels, we can convert image information into a table.

In [ ]:
props = measure.regionprops_table(
    labels,
    intensity_image=microscopy,
    properties=[
        "label",
        "area",
        "centroid",
        "mean_intensity",
        "max_intensity",
    ],
)

objects = pd.DataFrame(props)
objects


## 11. Load a real TIFF image

Let's apply the same workflow to real microscopy data: two fluorescence channels of a brain slice, each stored as a single 16-bit TIFF plane (5107 × 4012 pixels).

The steps below:

- **11.1** Load the two channels
- **11.2** Adjust the display contrast manually (LUT limits)
- **11.3** Display the green and red channels together
- **11.4** Register the red channel to the green channel
- **11.5** Apply the correction and display the aligned overlay
- **11.6** Segment the green channel by thresholding
- **11.7** Create the binary mask for the green channel
- **11.8** Segment the red channel
- **11.9** Improve thresholding: rolling-ball background subtraction
- **11.10** Repeat thresholding on the corrected image
- **11.11** Pixel-wise colocalization
- **11.12** Connected-component analysis
- **11.13** Measure morphological features with `regionprops`
- **11.14** Object-wise colocalization


### 11.1 Load the two channels

We load two TIFF files from the same acquisition of a brain slice:

- `green.tif` — channel 0 (virus), GFP, which we will display in **green**
- `red.tif` — channel 1 (auto-fluorescence), which we will display in **red**

Both files are therefore single 2-D planes: same brain, one time point, one Z position, two different channels. After loading, recall from Sections 1 and 2 that you can inspect the arrays with `.shape`, `.dtype`, `np.min`, and `np.max` — the images are 16-bit, so values can range from 0 to 65535.


In [ ]:
image_green = tifffile.imread("green.tif")
image_red = tifffile.imread("red.tif")


### 11.2 Adjust the display contrast manually (LUT limits)

Real fluorescence images are often dim, with only a small fraction of pixels carrying signal. Displaying them with the full data range — the matplotlib default — can make the sample look almost black.

The cell below shows the green channel two ways:

- **left** — the original display, using the full intensity range,
- **right** — the same data displayed with adjustable LUT limits (`vmin` / `vmax`).

Pixels darker than `vmin` are shown black, and pixels brighter than `vmax` are shown white. The slider is initialized from the 1st and 99.5th percentiles, a robust starting point when a few very bright pixels would otherwise flatten the rest of the image.

Use the slider to find limits that make the structures most informative. As in Section 4, this changes only the **display**, not the underlying pixel values.


In [ ]:
# Image intensity range
data_min = float(np.min(image_green))
data_max = float(np.max(image_green))

# Initial contrast limits (percentile-based)
p_low, p_high = np.percentile(image_green, [1, 99.5])

# Handle images with very sparse signal
if p_high <= p_low:
    p_low, p_high = data_min, data_max

# Two-handle LUT slider
lut_slider = widgets.FloatRangeSlider(
    value=(float(p_low), float(p_high)),
    min=data_min,
    max=data_max,
    step=(data_max - data_min) / 1000,
    description="LUT limits:",
    continuous_update=False,
    readout_format=".1f",
    layout=widgets.Layout(width="650px"),
)

def show_image(limits):
    vmin, vmax = limits

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Original image
    axes[0].imshow(image_green, cmap="gray")
    axes[0].set_title("Original display")

    # Image with adjustable LUT
    axes[1].imshow(
        image_green,
        cmap="gray",
        vmin=vmin,
        vmax=vmax
    )
    axes[1].set_title(f"LUT: {vmin:.1f} - {vmax:.1f}")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

widgets.interact(show_image, limits=lut_slider);


### 11.3 Display the green and red channels together

Let's compare the channels by overlaying them as a color composite: the green channel fills the green plane of an RGB image, the red channel fills the red plane, and pixels where both channels are bright appear **yellow**.

To mimic a real registration problem, the red image is first shifted with `np.roll`:

```text
image_red_shifted = np.roll(image_red, shift=(...), axis=(0, 1))
```

**Observe:** if you adjust the two LUT sliders, you will notice that the red channel is shifted with respect to the green channel — structures that should coincide appear translated. This is exactly the problem that registration solves, and we fix it in the next steps.


In [ ]:
# Display helper: map intensities from a (vmin, vmax) LUT range to [0, 1]
def apply_lut(image, limits):
    vmin, vmax = limits
    image = image.astype(np.float32)
    if vmax <= vmin:
        vmax = vmin + 1
    return np.clip((image - vmin) / (vmax - vmin), 0, 1)

# Two-handle LUT slider initialized from robust percentiles
def make_lut_slider(image, description):
    data_min = float(np.min(image))
    data_max = float(np.max(image))
    p_low, p_high = np.percentile(image, [1, 99.5])
    if p_high <= p_low:
        p_low, p_high = data_min, data_max
    return widgets.FloatRangeSlider(
        value=(float(p_low), float(p_high)),
        min=data_min,
        max=data_max,
        step=(data_max - data_min) / 1000,
        description=description,
        continuous_update=False,
        readout_format=".1f",
        layout=widgets.Layout(width="650px"),
    )

# One LUT slider per channel
green_slider = make_lut_slider(image_green, "Green LUT:")
red_slider = make_lut_slider(image_red, "Red LUT:")


image_red_shifted = np.roll(
    image_red,
    shift=(-40, 80),
    axis=(0, 1)
)

def show_overlay(green_limits, red_limits):

    green = apply_lut(image_green, green_limits)
    red = apply_lut(image_red_shifted, red_limits)

    # Combine channels into an RGB image
    rgb = np.zeros((*green.shape, 3), dtype=np.float32)
    rgb[..., 0] = red      # R
    rgb[..., 1] = green    # G
    rgb[..., 2] = 0        # B

    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(rgb)
    ax.set_title("Red + Green overlay", fontsize=15)
    ax.axis("off")
    plt.show()

output = widgets.interactive_output(
    show_overlay,
    {
        "green_limits": green_slider,
        "red_limits": red_slider
    }
)

display(widgets.VBox([green_slider, red_slider]), output)


### 11.4 Register the red channel to the green channel

We now estimate the translation that maps the shifted red image back onto the green image:

- the **green** image is the *reference* and stays fixed,
- the shifted **red** image is the *moving* image.

`skimage.registration.phase_cross_correlation` compares the Fourier transforms of the two images and returns the translation that best aligns them; with `upsample_factor=20` the estimate has sub-pixel precision. The estimated correction moves the **moving** image back onto the reference, so its sign is approximately the opposite of the shift that was applied.


In [ ]:
# Red image is the moving image
moving = image_red_shifted

# Green image is the reference
reference = image_green

# Estimate the translation
recovered_shift, error, phasediff = registration.phase_cross_correlation(
    reference,
    moving,
    upsample_factor=20
)

print("Estimated correction (y, x):", recovered_shift)


### 11.5 Apply the correction and display the aligned overlay

The estimated translation is applied to the red image with `scipy.ndimage.shift`. Bilinear interpolation (`order=1`) fills in non-integer sample positions, and `mode="grid-wrap"` handles values that shift in from beyond the border.

Run the cell and compare the overlay with the uncorrected one from 11.3: structures that appeared as separate red and green fringes should now largely coincide as yellow.


In [ ]:
# Apply the estimated correction to the original red image
registered_red = ndi.shift(
    image_red_shifted,
    shift=recovered_shift,
    order=1,
    mode="grid-wrap"
)

def show_overlay(green_limits, red_limits):

    green = apply_lut(image_green, green_limits)
    red = apply_lut(registered_red, red_limits)

    # Combine channels into an RGB image
    rgb = np.zeros((*green.shape, 3), dtype=np.float32)
    rgb[..., 0] = red      # R
    rgb[..., 1] = green    # G
    rgb[..., 2] = 0        # B

    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(rgb)
    ax.set_title("Red + Green overlay", fontsize=15)
    ax.axis("off")
    plt.show()

output = widgets.interactive_output(
    show_overlay,
    {
        "green_limits": green_slider,
        "red_limits": red_slider
    }
)

display(widgets.VBox([green_slider, red_slider]), output)


### 11.6 Segment the green channel by thresholding

We now segment the green channel using the approach from Section 7. The cell below provides three sliders:

- **LUT** — display limits for the grayscale image (display only),
- **Threshold** — the intensity above which a pixel counts as foreground,
- **Mask alpha** — transparency of the red mask overlay.

The threshold is always applied to the **raw** image data, independent of the LUT choice. The printed feedback reports how many pixels, and what fraction of the image, lie above the threshold.

Use the sliders to choose a threshold that captures the labeled structures while excluding the background, and note the value you settle on for the next cell.


In [ ]:
# Choose the image to segment
image = image_green   # or image_red, or any 2D grayscale image

# Basic checks
assert image.ndim == 2, "This example expects a 2D grayscale image."

data_min = float(np.min(image))
data_max = float(np.max(image))

# Initial LUT limits from percentiles
lut_low, lut_high = np.percentile(image, [1, 99.5])
if lut_high <= lut_low:
    lut_low, lut_high = data_min, data_max

# Initial threshold (you can change this choice)
thr_init = float(np.percentile(image, 95))

# LUT slider
lut_slider = widgets.FloatRangeSlider(
    value=(float(lut_low), float(lut_high)),
    min=data_min,
    max=data_max,
    step=max((data_max - data_min) / 1000, 1e-6),
    description="LUT:",
    continuous_update=False,
    readout_format=".1f",
    layout=widgets.Layout(width="650px"),
)

# Threshold slider
threshold_slider = widgets.FloatSlider(
    value=thr_init,
    min=data_min,
    max=data_max,
    step=max((data_max - data_min) / 1000, 1e-6),
    description="Threshold:",
    continuous_update=False,
    readout_format=".1f",
    layout=widgets.Layout(width="650px"),
)

# Opacity slider for the mask overlay (optional, but useful)
alpha_slider = widgets.FloatSlider(
    value=0.35,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Mask alpha:",
    continuous_update=False,
    readout_format=".2f",
    layout=widgets.Layout(width="650px"),
)

def show_thresholding(lut_limits, threshold, mask_alpha):
    vmin, vmax = lut_limits

    # Threshold is applied to the raw image data
    mask = image >= threshold

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Left: original image with chosen LUT
    axes[0].imshow(image, cmap="gray", vmin=vmin, vmax=vmax)
    axes[0].set_title("Original image")
    axes[0].axis("off")

    # Right: original image + mask overlay
    axes[1].imshow(image, cmap="gray", vmin=vmin, vmax=vmax)
    axes[1].imshow(mask, cmap="Reds", alpha=mask_alpha, interpolation="none")
    axes[1].set_title(f"Thresholded mask overlay (threshold = {threshold:.1f})")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

    # Useful quantitative feedback for students
    n_pixels = mask.sum()
    frac = 100 * n_pixels / mask.size
    print(f"Pixels above threshold: {n_pixels} / {mask.size} ({frac:.2f}%)")

output = widgets.interactive_output(
    show_thresholding,
    {
        "lut_limits": lut_slider,
        "threshold": threshold_slider,
        "mask_alpha": alpha_slider,
    }
)

display(widgets.VBox([lut_slider, threshold_slider, alpha_slider]), output)


### Try it yourself

Input the value of the threshold that you determined using the slider

In [ ]:
### YOUR CODE HERE
# threshold_green = 


### 11.7 Create the binary mask for the green channel

The comparison `image_green > threshold_green` classifies every pixel and returns a **boolean mask**: `True` for foreground, `False` for background. A mask contains no intensity information — only the segmentation decision (Section 7).

In [ ]:
mask_green = image_green > threshold_green

### Try it yourself

Display the `mask_green` and compare the displayed mask with the interactive overlay from 11.6: they should agree.
Use `plt.imshow()` function with the `grey` colormap

In [ ]:
# YOUR CODE HERE


### 11.8 Try to segment the red channel - and see why it's hard.

The same thresholding procedure is now applied to the red channel, with the same interactive sliders.

Real red channels often carry more uneven background and autofluorescence than the green channel, so a single global threshold may be a poor fit here — which motivates the background-correction step in 11.9.


In [ ]:
# Choose the image to segment
image = image_red

# Basic checks
assert image.ndim == 2, "This example expects a 2D grayscale image."

data_min = float(np.min(image))
data_max = float(np.max(image))

# Initial LUT limits from percentiles
lut_low, lut_high = np.percentile(image, [1, 99.5])
if lut_high <= lut_low:
    lut_low, lut_high = data_min, data_max

# Initial threshold (you can change this choice)
thr_init = float(np.percentile(image, 95))

# LUT slider
lut_slider = widgets.FloatRangeSlider(
    value=(float(lut_low), float(lut_high)),
    min=data_min,
    max=data_max,
    step=max((data_max - data_min) / 1000, 1e-6),
    description="LUT:",
    continuous_update=False,
    readout_format=".1f",
    layout=widgets.Layout(width="650px"),
)

# Threshold slider
threshold_slider = widgets.FloatSlider(
    value=thr_init,
    min=data_min,
    max=data_max,
    step=max((data_max - data_min) / 1000, 1e-6),
    description="Threshold:",
    continuous_update=False,
    readout_format=".1f",
    layout=widgets.Layout(width="650px"),
)

# Opacity slider for the mask overlay (optional, but useful)
alpha_slider = widgets.FloatSlider(
    value=0.35,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Mask alpha:",
    continuous_update=False,
    readout_format=".2f",
    layout=widgets.Layout(width="650px"),
)

def show_thresholding(lut_limits, threshold, mask_alpha):
    vmin, vmax = lut_limits

    # Threshold is applied to the raw image data
    mask = image >= threshold

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Left: original image with chosen LUT
    axes[0].imshow(image, cmap="gray", vmin=vmin, vmax=vmax)
    axes[0].set_title("Original image")
    axes[0].axis("off")

    # Right: original image + mask overlay
    axes[1].imshow(image, cmap="gray", vmin=vmin, vmax=vmax)
    axes[1].imshow(mask, cmap="Reds", alpha=mask_alpha, interpolation="none")
    axes[1].set_title(f"Thresholded mask overlay (threshold = {threshold:.1f})")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

    # Useful quantitative feedback for students
    n_pixels = mask.sum()
    frac = 100 * n_pixels / mask.size
    print(f"Pixels above threshold: {n_pixels} / {mask.size} ({frac:.2f}%)")

output = widgets.interactive_output(
    show_thresholding,
    {
        "lut_limits": lut_slider,
        "threshold": threshold_slider,
        "mask_alpha": alpha_slider,
    }
)

display(widgets.VBox([lut_slider, threshold_slider, alpha_slider]), output)


### 11.9 Improve thresholding: rolling-ball background subtraction (slow step!)

A common preprocessing step before thresholding is to remove a slowly varying background. The **rolling-ball** algorithm (as in ImageJ) slides a ball of a given diameter along the underside of the intensity surface; the ball cannot enter narrow peaks, so it traces the smooth background beneath them. Subtracting this background leaves the signal on a flat baseline.

The cell below shows three panels:

- **original** fluorescence,
- the **estimated background** traced by the ball,
- the **background-subtracted** image, `max(raw − background, 0)`.

The corrected image and the estimated background are stored in the `results` dictionary for use in later cells. Try different ball diameters (the slider ranges from 10 to 100 px, default 40): a small ball follows more structure and removes more signal, while a large ball leaves more background. Choose a diameter that flattens the background while preserving the labeled structures.


In [ ]:
# Corrected version of the cell above: widget names are unique (rb_*) so that
# re-running the thresholding cells (which reuse the name lut_slider) can no
# longer hijack the LUT limits of this widget.

# -------------------------------------------------------
# INPUT IMAGE
# -------------------------------------------------------

image = np.asarray(image_red, dtype=np.float32)

if image.ndim != 2:
    raise ValueError("Please provide a 2D grayscale image.")

# Use a smaller ROI for faster interactive processing.
# Set to None to process the entire image.
PREVIEW_SIZE = None   # 320

if PREVIEW_SIZE is not None:
    h, w = image.shape
    y0 = max(0, (h - PREVIEW_SIZE) // 2)
    x0 = max(0, (w - PREVIEW_SIZE) // 2)
    image = image[y0:y0 + PREVIEW_SIZE,
                  x0:x0 + PREVIEW_SIZE]

# Linear normalization to 0-255
raw_min = float(image.min())
raw_range = max(float(image.max() - raw_min), 1e-10)
raw = (image - raw_min) * (255.0 / raw_range)

green_cmap = LinearSegmentedColormap.from_list(
    "fluorescence", ["black", "lime"]
)

# -------------------------------------------------------
# ROLLING-BALL BACKGROUND
# -------------------------------------------------------

@lru_cache(maxsize=10)
def get_background(diameter):
    return rolling_ball(raw, radius=diameter / 2.0)

# -------------------------------------------------------
# WIDGETS
# -------------------------------------------------------

lut_lo = int(np.percentile(raw, 1))
lut_hi = min(
    255,
    max(lut_lo + 1, int(np.percentile(raw, 99.5)))
)

rb_lut_slider = widgets.IntRangeSlider(
    value=(lut_lo, lut_hi),
    min=0,
    max=255,
    step=1,
    description="Original LUT:",
    continuous_update=False,
    layout=widgets.Layout(width="550px"),
    style={"description_width": "150px"}
)

rb_diameter_slider = widgets.IntSlider(
    value=40,
    min=10,
    max=100,
    step=10,
    description="Ball diameter (px):",
    continuous_update=False,
    layout=widgets.Layout(width="550px"),
    style={"description_width": "150px"}
)

rb_image_widget = widgets.Image(format="png")

# -------------------------------------------------------
# DISPLAY CONTRAST
# -------------------------------------------------------

def auto_lut(arr, low=1, high=99.8):
    vmin = float(np.percentile(arr, low))
    vmax = float(np.percentile(arr, high))

    if vmax <= vmin:
        vmax = vmin + 1

    return vmin, vmax

# -------------------------------------------------------
# UPDATE FUNCTION
# -------------------------------------------------------

results = {}

def update(change=None):

    diameter = rb_diameter_slider.value
    raw_vmin, raw_vmax = rb_lut_slider.value

    if raw_vmax <= raw_vmin:
        raw_vmax = raw_vmin + 1

    # Rolling-ball background correction
    background = get_background(diameter)
    corrected = np.maximum(raw - background, 0)
    
    # Save the latest results for use in other notebook cells
    results["corrected"] = corrected
    results["background"] = background
    results["diameter"] = diameter
    
    # Independent display LUTs
    bg_vmin, bg_vmax = auto_lut(background)
    corr_vmin, corr_vmax = auto_lut(corrected)

    # Create three-panel figure
    with plt.ioff():
        fig, ax = plt.subplots(
            1, 3,
            figsize=(13, 4.5),
            constrained_layout=True
        )

    # Original
    ax[0].imshow(
        raw,
        cmap=green_cmap,
        vmin=raw_vmin,
        vmax=raw_vmax
    )
    ax[0].set_title("Original fluorescence")

    # Estimated background
    ax[1].imshow(
        background,
        cmap=green_cmap,
        vmin=bg_vmin,
        vmax=bg_vmax
    )
    ax[1].set_title("Estimated background")

    # Background-subtracted
    ax[2].imshow(
        corrected,
        cmap=green_cmap,
        vmin=corr_vmin,
        vmax=corr_vmax
    )
    ax[2].set_title("Background-subtracted")

    for a in ax:
        a.axis("off")

    fig.suptitle(
        f"Rolling-ball diameter: {diameter} px",
        fontsize=14
    )

    # Render figure to PNG
    buf = io.BytesIO()
    fig.savefig(
        buf,
        format="png",
        dpi=90,
        facecolor=fig.get_facecolor()
    )

    plt.close(fig)

    rb_image_widget.value = buf.getvalue()
    buf.close()

# -------------------------------------------------------
# CONNECT WIDGETS
# -------------------------------------------------------

rb_lut_slider.observe(update, names="value")
rb_diameter_slider.observe(update, names="value")

display(widgets.VBox([
    rb_lut_slider,
    rb_diameter_slider,
    rb_image_widget
]))

update()


### 11.10 Repeat thresholding on the corrected red channel

Now threshold the background-corrected image stored in `results["corrected"]` (from 11.9), using the same interactive sliders.

Compare the threshold value and the resulting mask with the uncorrected red channel from 11.8: background correction usually allows a lower, more reliable threshold, because the background baseline is now flat.


In [ ]:
# Choose the image to segment
image = results["corrected"]

# Basic checks
assert image.ndim == 2, "This example expects a 2D grayscale image."

data_min = float(np.min(image))
data_max = float(np.max(image))

# Initial LUT limits from percentiles
lut_low, lut_high = np.percentile(image, [1, 99.5])
if lut_high <= lut_low:
    lut_low, lut_high = data_min, data_max

# Initial threshold (you can change this choice)
thr_init = float(np.percentile(image, 95))

# LUT slider
lut_slider = widgets.FloatRangeSlider(
    value=(float(lut_low), float(lut_high)),
    min=data_min,
    max=data_max,
    step=max((data_max - data_min) / 1000, 1e-6),
    description="LUT:",
    continuous_update=False,
    readout_format=".1f",
    layout=widgets.Layout(width="650px"),
)

# Threshold slider
threshold_slider = widgets.FloatSlider(
    value=thr_init,
    min=data_min,
    max=data_max,
    step=max((data_max - data_min) / 1000, 1e-6),
    description="Threshold:",
    continuous_update=False,
    readout_format=".1f",
    layout=widgets.Layout(width="650px"),
)

# Opacity slider for the mask overlay (optional, but useful)
alpha_slider = widgets.FloatSlider(
    value=0.35,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Mask alpha:",
    continuous_update=False,
    readout_format=".2f",
    layout=widgets.Layout(width="650px"),
)

def show_thresholding(lut_limits, threshold, mask_alpha):
    vmin, vmax = lut_limits

    # Threshold is applied to the raw image data
    mask = image >= threshold

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Left: original image with chosen LUT
    axes[0].imshow(image, cmap="gray", vmin=vmin, vmax=vmax)
    axes[0].set_title("Original image")
    axes[0].axis("off")

    # Right: original image + mask overlay
    axes[1].imshow(image, cmap="gray", vmin=vmin, vmax=vmax)
    axes[1].imshow(mask, cmap="Reds", alpha=mask_alpha, interpolation="none")
    axes[1].set_title(f"Thresholded mask overlay (threshold = {threshold:.1f})")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

    # Useful quantitative feedback for students
    n_pixels = mask.sum()
    frac = 100 * n_pixels / mask.size
    print(f"Pixels above threshold: {n_pixels} / {mask.size} ({frac:.2f}%)")

output = widgets.interactive_output(
    show_thresholding,
    {
        "lut_limits": lut_slider,
        "threshold": threshold_slider,
        "mask_alpha": alpha_slider,
    }
)

display(widgets.VBox([lut_slider, threshold_slider, alpha_slider]), output)


### Try it yourself

Specify the threshold from the backgound-subtracted red channel image (which you found using the slider in the cell above)

In [ ]:
# YOUR CODE HERE
# threshold_red = 

In [ ]:
# apply the threshold to get the mask of the red channel
mask_red = results["corrected"] > threshold_red

### Try it yourself

Display the mask with `gray` colormap

In [ ]:
# YOUR CODE HERE

### 11.11 Pixel-wise colocalization

Colocalization asks how much of the signal in the two channels occupies the same pixels. Using the binary masks created above (`mask_green` from 11.7, `mask_red` from 11.10), the cell below computes three sets of regions:

- `green_only = mask_green & ~mask_red`
- `red_only = mask_red & ~mask_green`
- `overlap = mask_green & mask_red`

and from them reports standard metrics:

- the **fraction of green overlapping red**, and vice versa,
- the **Jaccard index** (intersection over union),
- the **Dice coefficient**.

The middle panel shows the thresholded channels with their overlap in yellow; the right panel marks overlapping pixels in white. The display uses the registered red channel from 11.5.


In [ ]:
# ----------------------------------------------------
# 1. Threshold both channels
# ----------------------------------------------------

# Done above

# ----------------------------------------------------
# 2. Calculate co-localization
# ----------------------------------------------------

green_only = mask_green & ~mask_red
red_only = mask_red & ~mask_green
overlap = mask_green & mask_red

# ----------------------------------------------------
# 3. Calculate co-localization metrics
# ----------------------------------------------------

n_green = np.count_nonzero(mask_green)
n_red = np.count_nonzero(mask_red)
n_overlap = np.count_nonzero(overlap)
n_union = np.count_nonzero(mask_green | mask_red)

# Fraction of each channel overlapping the other
fraction_green = n_overlap / n_green if n_green else np.nan
fraction_red = n_overlap / n_red if n_red else np.nan

# Intersection over Union (Jaccard index)
jaccard = n_overlap / n_union if n_union else np.nan

# Dice coefficient
dice = (
    2 * n_overlap / (n_green + n_red)
    if n_green + n_red else np.nan
)

# ----------------------------------------------------
# 4. Prepare the original RGB image for display
# ----------------------------------------------------

def normalize_display(image):
    p1, p99 = np.percentile(image, [1, 99.5])
    if p99 <= p1:
        return np.zeros(image.shape, dtype=np.float32)

    image = image.astype(np.float32)
    return np.clip((image - p1) / (p99 - p1), 0, 1)

rgb = np.zeros((*image_green.shape, 3), dtype=np.float32)
rgb[..., 0] = normalize_display(registered_red)
rgb[..., 1] = normalize_display(image_green)

# Color-coded binary masks:
# Red only = red
# Green only = green
# Both = yellow
seg_rgb = np.zeros_like(rgb)
seg_rgb[..., 0] = mask_red.astype(float)
seg_rgb[..., 1] = mask_green.astype(float)

# White transparent overlay marking overlapping pixels
overlap_overlay = np.zeros((*overlap.shape, 4), dtype=np.float32)
overlap_overlay[..., :3] = 1.0
overlap_overlay[..., 3] = overlap.astype(float) * 0.8

# ----------------------------------------------------
# 5. Visualize results
# ----------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(rgb)
axes[0].set_title("Original fluorescence")

axes[1].imshow(seg_rgb)
axes[1].set_title("Thresholded channels\nYellow = overlap")

axes[2].imshow(rgb)
axes[2].imshow(overlap_overlay)
axes[2].set_title("Co-localization\nWhite = overlapping pixels")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

# ----------------------------------------------------
# 6. Print quantitative results
# ----------------------------------------------------

print(f"Green-positive pixels: {n_green:,}")
print(f"Red-positive pixels:   {n_red:,}")
print(f"Overlapping pixels:    {n_overlap:,}")

print(f"\nGreen signal overlapping red: {fraction_green:.1%}")
print(f">>>>>>>>> Red signal overlapping green: {fraction_red:.1%}")
print(f"Jaccard index: {jaccard:.3f}")
print(f"Dice coefficient: {dice:.3f}")


### 11.12 Connected-component analysis

A binary mask says only foreground/background. As in Section 9, connected-component labeling with `measure.label` (`connectivity=2`, i.e. 8-connected in 2-D) turns each connected group of foreground pixels into an object with a unique integer ID.

The cells below:

1. label both channels and report the object counts,
2. define `label_and_filter`, which removes components smaller than `min_area` pixels and relabels the rest consecutively,
3. let you tune `min_area` interactively.

Sweep the **min area** slider and watch the object counts: small specks of noise disappear, while real structures survive.


In [ ]:
# --------------------------------------------------
# 1. Label connected components
# --------------------------------------------------

# connectivity=2 means 8-connected pixels in 2D
labels_green = measure.label(mask_green, connectivity=2)
labels_red = measure.label(mask_red, connectivity=2)

print(f"Green channel: {labels_green.max()} objects")
print(f"Red channel:   {labels_red.max()} objects")

# --------------------------------------------------
# 2. Normalize image intensity for visualization
# --------------------------------------------------

def normalize_display(image):
    p1, p99 = np.percentile(image, [1, 99.5])

    if p99 <= p1:
        return np.zeros(image.shape, dtype=np.float32)

    image = image.astype(np.float32)
    return np.clip((image - p1) / (p99 - p1), 0, 1)

# --------------------------------------------------
# 3. Overlay labels onto original images
# --------------------------------------------------

overlay_green = color.label2rgb(
    labels_green,
    image=normalize_display(image_green),
    alpha=0.4,
    bg_label=0,
    kind="overlay"
)

overlay_red = color.label2rgb(
    labels_red,
    image=normalize_display(registered_red),
    alpha=0.4,
    bg_label=0,
    kind="overlay"
)

# --------------------------------------------------
# 4. Visualize
# --------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(overlay_green)
axes[0].set_title(
    f"Green channel: {labels_green.max()} objects",
    fontsize=15
)

axes[1].imshow(overlay_red)
axes[1].set_title(
    f"Red channel: {labels_red.max()} objects",
    fontsize=15
)

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


That's a lot of objects for a tiny image! Looks like we picked up many tiny noise peaks.

In [ ]:
def label_and_filter(mask, min_area=20):
    # Identify all connected components
    labels = measure.label(mask, connectivity=2)

    # Count pixels in every labeled component
    areas = np.bincount(labels.ravel())

    # Keep only components above minimum area
    keep = areas >= min_area
    keep[0] = False  # Exclude background

    filtered_mask = keep[labels]

    # Relabel components consecutively
    return measure.label(filtered_mask, connectivity=2)


# LUT (contrast) sliders for the displayed images
green_lut_slider = make_lut_slider(image_green, "Green LUT:")
red_lut_slider = make_lut_slider(registered_red, "Red LUT:")


def show_components(min_area, green_limits, red_limits):
    labels_g = label_and_filter(mask_green, min_area)
    labels_r = label_and_filter(mask_red, min_area)

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    for ax, image, labels, limits, name in zip(
        axes,
        [image_green, registered_red],
        [labels_g, labels_r],
        [green_limits, red_limits],
        ["Green", "Red"]
    ):
        overlay = color.label2rgb(
            labels,
            image=apply_lut(image, limits),
            alpha=0.4,
            bg_label=0
        )
        # label2rgb blending can leave values a few float32 ulps above 1;
        # clip so imshow does not warn about RGB data outside [0, 1]
        overlay = np.clip(overlay, 0, 1)

        ax.imshow(overlay)
        ax.set_title(
            f"{name}: {labels.max()} objects",
            fontsize=15
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()

widgets.interact(
    show_components,
    min_area=widgets.IntSlider(
        value=20,
        min=1,
        max=500,
        step=1,
        description="Min area:",
        continuous_update=False,
        layout=widgets.Layout(width="600px")
    ),
    green_limits=green_lut_slider,
    red_limits=red_lut_slider
);


### Try it yourself

Specify the min_area that gives best results below

In [ ]:
# YOUR CODE HERE
min_area = 20

In [ ]:
labels_green = label_and_filter(mask_green, min_area=min_area)
labels_red = label_and_filter(mask_red, min_area=min_area)

### 11.13 Measure morphological features with `regionprops`

Once objects have labels, `measure.regionprops_table` converts the segmentation into a table of per-object measurements (Section 10). The properties measured here include:

- **area** — number of pixels in the object,
- **centroid** — the object's center position (y, x),
- **perimeter**, **eccentricity**, **solidity**, **extent** — shape descriptors,
- **major/minor axis length**, **equivalent diameter** — size descriptors.

The tables are sorted by area, largest first, for both channels


In [ ]:
# 10 measurements (centroid contributes 2 columns)
properties = [
    "label",
    "area",
    "centroid",
    "perimeter",
    "eccentricity",
    "solidity",
    "extent",
    "axis_major_length",
    "axis_minor_length",
    "equivalent_diameter_area",
]

def make_regionprops_table(labels):
    # Extract properties
    df = pd.DataFrame(
        measure.regionprops_table(
            labels,
            properties=properties
        )
    )

    # Rename columns for readability
    df = df.rename(columns={
        "label": "Object ID",
        "area": "Area (px²)",
        "centroid-0": "Centroid Y",
        "centroid-1": "Centroid X",
        "perimeter": "Perimeter (px)",
        "eccentricity": "Eccentricity",
        "solidity": "Solidity",
        "extent": "Extent",
        "axis_major_length": "Major axis (px)",
        "axis_minor_length": "Minor axis (px)",
        "equivalent_diameter_area": "Equivalent diameter (px)"
    })

    # Sort largest to smallest
    df = df.sort_values(
        "Area (px²)", ascending=False
    ).reset_index(drop=True)

    return df

# Generate tables for both channels
df_green = make_regionprops_table(labels_green)
df_red = make_regionprops_table(labels_red)

# Display largest 10 objects from each channel
print(f"GREEN CHANNEL: {len(df_green)} objects")
display(df_green.head(10).round(2))

print(f"\nRED CHANNEL: {len(df_red)} objects")
display(df_red.head(10).round(2))


### 11.14 Object-wise colocalization

Pixel-wise colocalization says nothing about individual objects. In object-based colocalization, overlap is evaluated **per object**: for every labeled object in each channel we count

- how many of its pixels overlap the other channel,
- what percentage of its area that represents,
- how many distinct partner objects it touches.

An object is flagged `Colocalized` when at least `min_pixels = 5` of its pixels overlap **and** at least `min_fraction = 0.20` (20 %) of its area overlaps. The two channels are evaluated independently, and the pairs table lists every overlapping green–red object pair.

The final cells highlight the co-localized objects on the composite image. Compare these object-based results with the pixel-wise metrics from 11.11: they answer related but different questions.


In [ ]:
def object_colocalization(
    labels_green,
    labels_red,
    min_fraction=0.20,
    min_pixels=5,
):
    """
    Analyze object-wise overlap between two labeled images.

    An object is considered co-localized when:
      1. At least min_pixels of its pixels overlap
      2. At least min_fraction of its area overlaps

    The two channels are evaluated independently.
    Background label 0 is ignored.

    Returns:
        green_df, red_df, pairs_df
    """

    if labels_green.shape != labels_red.shape:
        raise ValueError("Images must have the same shape")

    ng = int(labels_green.max())
    nr = int(labels_red.max())

    # Area of each object (number of pixels)
    area_g = np.bincount(
        labels_green.ravel(), minlength=ng + 1
    )
    area_r = np.bincount(
        labels_red.ravel(), minlength=nr + 1
    )

    # Pixels belonging to objects in both channels
    shared = (labels_green > 0) & (labels_red > 0)

    # Overlapping pixels for every object
    overlap_g = np.bincount(
        labels_green[shared], minlength=ng + 1
    )
    overlap_r = np.bincount(
        labels_red[shared], minlength=nr + 1
    )

    # Identify every overlapping green/red object pair
    # Encode the label pair into a single integer
    packed = (
        labels_green[shared].astype(np.int64) * (nr + 1)
        + labels_red[shared]
    )

    keys, counts = np.unique(packed, return_counts=True)

    gids = keys // (nr + 1)
    rids = keys % (nr + 1)

    # Table of all intersecting object pairs
    pairs_df = pd.DataFrame({
        "Green ID": gids,
        "Red ID": rids,
        "Overlap (px)": counts,
        "% Green covered": 100 * counts / area_g[gids],
        "% Red covered": 100 * counts / area_r[rids],
    })

    pairs_df = pairs_df.sort_values(
        "Overlap (px)", ascending=False
    ).reset_index(drop=True)

    # Number of distinct overlapping partners
    partners_g = np.bincount(gids, minlength=ng + 1)
    partners_r = np.bincount(rids, minlength=nr + 1)

    # Create per-object tables
    def make_table(areas, overlaps, partners):
        ids = np.arange(1, len(areas))

        df = pd.DataFrame({
            "Object ID": ids,
            "Area (px)": areas[1:],
            "Overlap (px)": overlaps[1:],
            "Overlap (%)": (
                100 * overlaps[1:] / areas[1:]
            ),
            "Partner objects": partners[1:],
        })

        df["Colocalized"] = (
            (df["Overlap (px)"] >= min_pixels)
            & (df["Overlap (%)"] >= 100 * min_fraction)
        )

        return df.sort_values(
            "Area (px)", ascending=False
        ).reset_index(drop=True)

    green_df = make_table(area_g, overlap_g, partners_g)
    red_df = make_table(area_r, overlap_r, partners_r)

    return green_df, red_df, pairs_df

# --------------------------------------------------
# Run the analysis
# --------------------------------------------------

green_coloc, red_coloc, pairs_coloc = object_colocalization(
    labels_green,
    labels_red,
    min_fraction=0.20,
    min_pixels=5
)

print("OVERLAPPING RED-GREEN PAIRS")
display(pairs_coloc.head(20).round(2))


In [ ]:
def print_colocalization_summary(green_df, red_df):

    n_green = len(green_df)
    n_red = len(red_df)

    coloc_green = int(green_df["Colocalized"].sum())
    coloc_red = int(red_df["Colocalized"].sum())

    print("OBJECT-WISE CO-LOCALIZATION SUMMARY")
    print("-----------------------------------")

    print(f"Green objects: {n_green}")
    print(f"Red objects:   {n_red}")

    print(
        f"\nGreen objects co-localized with red: "
        f"{coloc_green}/{n_green} "
        f"({100 * coloc_green / n_green:.1f}%)"
        if n_green else "\nNo green objects detected"
    )

    print(
        f">>>>>>>>>>> Red objects co-localized with green: "
        f"{coloc_red}/{n_red} "
        f"({100 * coloc_red / n_red:.1f}%)"
        if n_red else "\nNo red objects detected"
    )

print_colocalization_summary(green_coloc, red_coloc)


In [ ]:
# Create RGB composite using the original intensities
rgb = np.zeros((*image_green.shape, 3), dtype=np.float32)
rgb[..., 0] = normalize_display(registered_red)
rgb[..., 1] = normalize_display(image_green)

# IDs of objects classified as co-localized
red_ids = red_coloc.loc[
    red_coloc["Colocalized"], "Object ID"
].to_numpy()

green_ids = green_coloc.loc[
    green_coloc["Colocalized"], "Object ID"
].to_numpy()

# Masks of entire co-localized objects
matched_red = np.isin(labels_red, red_ids)
matched_green = np.isin(labels_green, green_ids)

# Helper to make transparent overlays
def make_overlay(mask, color, alpha=0.55):
    overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
    overlay[..., :3] = color
    overlay[..., 3] = mask.astype(float) * alpha
    return overlay

# Create overlays
overlay_red = make_overlay(matched_red, (0, 0, 0))      # black

# Plot only the 2 requested panels
fig, axes = plt.subplots(1, 1, figsize=(10, 10))

axes.imshow(rgb, alpha=0.5)
axes.imshow(overlay_red)
axes.set_title("Co-localized red objects (black)", fontsize=12)
axes.axis("off")

plt.tight_layout()
plt.show()

## 12 (optional) Explore the brain in 3D
- Load the `green_3d.tif` and `red_3d.tif` images
- Display 3 orthogonal slices for one of them
- Try to adapt the 2D code to threshold the brain in 3D, and label the connected components in green and red channels
- Compute pixel-wise and object-wise co-localization metrics.


## Takeaway: good image-analysis habits

1. **Keep raw data unchanged.**
2. **Separate display operations from data-processing operations.**
3. **Record pixel size and other calibration metadata.**
4. **Know the array dimension order.**
5. **Record processing parameters.**
6. **Inspect intermediate results, not just the final numbers.**
7. **Use masks and overlays to validate segmentation visually.**
8. **Avoid interpreting display colors as intrinsic properties of the data.**
9. **Watch for saturation and clipping.**
10. **Save code and software versions for reproducibility.**

## Concept check

Discuss these questions:

1. If you change a grayscale image to a green LUT, have the pixel values changed?
2. Is a 16-bit image always more informative than an 8-bit image?
3. Is pixel size the same thing as optical resolution?
4. What is the difference between a binary mask and a labeled image?
5. Why can Gaussian filtering improve segmentation?
6. Why can too much Gaussian filtering be harmful?
7. What does a thresholding operation do?
8. Why does the connectivity definition matter?
9. What does registration change: intensities, spatial coordinates, or both?
10. Which steps in an image-analysis workflow should be reproducible?

---
## Advanced topics to look into

After this notebook, natural next lessons include:

- 3-D image stacks and maximum-intensity projections
- local/adaptive thresholding
- watershed segmentation
- background subtraction
- deconvolution
- 3-D colocalization
- object tracking
- 3-D connected components
- image metadata and OME-TIFF / OME-Zarr
- segmentation quality metrics
- machine-learning and deep-learning segmentation, classification etc